# Demo usage of the common interface for generating embeddings

In [1]:
import sys
import os
import pandas as pd
import numpy as np

# Add project root to sys.path to allow imports
sys.path.append("..")

from helpers.data_loaders import load_movielens_data, load_steam_data
from helpers.evaluation import leave_one_out_split, calculate_metrics, print_metrics
from src.models import (
    LightGCNGenerator,
    Node2VecGenerator,
    CleoraGenerator,
    MatrixFactorizationGenerator,
    NCFGenerator
)


## 1. Data preprocessing

In [2]:
# Load MovieLens data
movies_df, ratings_df = load_movielens_data("datasets/movies/movies.csv", "datasets/movies/ratings.csv")
movielens_interactions = pd.DataFrame({
    'user_id': 'ml_user_' + ratings_df['userId'].astype(str),
    'item_id': 'ml_item_' + ratings_df['movieId'].astype(str),
    'rating': ratings_df['rating'],
    'timestamp': ratings_df['timestamp']
}).drop_duplicates(subset=['user_id', 'item_id'])
 
# Preprocess MovieLens: Filter for high ratings to treat as positive interactions
movielens_interactions = movielens_interactions[movielens_interactions['rating'] >= 4.0].copy()

# Load Steam data
reviews_df_steam, items_df_steam = load_steam_data(
    "datasets/steam/formatted_user_reviews.json",
    "datasets/steam/formatted_steam_games.json"
)
steam_interactions = pd.DataFrame({
    'user_id': 'steam_user_' + reviews_df_steam['user_id'].astype(str),
    'item_id': 'steam_item_' + reviews_df_steam['app_id'].astype(str),
    'rating': 1.0, # Steam reviews are implicit feedback, use 1.0 for positive
    'timestamp': 0 # Default timestamp for Steam reviews (no date available)
}).drop_duplicates(subset=['user_id', 'item_id'])

# Concatenate and drop duplicates
all_interactions = pd.concat([movielens_interactions[['user_id', 'item_id', 'rating', 'timestamp']],
                              steam_interactions[['user_id', 'item_id', 'rating', 'timestamp']]])
all_interactions = all_interactions.drop_duplicates(subset=['user_id', 'item_id']).reset_index(drop=True)

# Sample 100,000 interactions if the dataset is larger
if len(all_interactions) > 500000:
    interactions_df = all_interactions.sample(n=500000, random_state=42).reset_index(drop=True)
else:
    interactions_df = all_interactions

print(f"All Unique users: {all_interactions['user_id'].nunique()}")
print(f"All Unique items: {all_interactions['item_id'].nunique()}")

print(f"Using {len(interactions_df)} interactions for demonstration.")
print(f"Unique users: {interactions_df['user_id'].nunique()}")
print(f"Unique items: {interactions_df['item_id'].nunique()}")

# Split Data (Leave-One-Out)
train_df, test_df = leave_one_out_split(interactions_df, time_col='timestamp')
print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")


All Unique users: 184419
All Unique items: 43660
Using 500000 interactions for demonstration.
Unique users: 123229
Unique items: 15558
Train size: 376771
Test size: 123229


## 2. Define Models

In [ ]:
models = [
    ("LightGCN", LightGCNGenerator(epochs=5, batch_size=128)),
    ("Node2Vec", Node2VecGenerator(epochs=100, batch_size=128)),
    ("MatrixFactorization", MatrixFactorizationGenerator(epochs=5, batch_size=128)),
    ("NCF", NCFGenerator(epochs=15, batch_size=128)),
    ("Cleora", CleoraGenerator(num_walks=2))
]

## 3. Train and Evaluate

In [ ]:
results_dir = 'models'
os.makedirs(results_dir, exist_ok=True)

for name, model in models:    
    print(f"\n--- Testing {name} ---")
    try:
        # Determine filename
        filename = f"partial_{name.lower()}_model.pth"
        
        load_path = os.path.join(results_dir, filename)
        loaded = False
        
        if os.path.exists(load_path):
            print(f"Found checkpoint at {load_path}. Attempting to load...")
            try:
                model.load(load_path)
                print("Model loaded successfully.")
                loaded = True
            except Exception as e:
                print(f"Failed to load {name} (will retrain): {e}")
        else:
            print(f"No checkpoint found at {load_path}. Will train from scratch.")
        
        if not loaded:
            print(f"Training {name}...")
            model.fit(train_df, user_col='user_id', item_col='item_id', rating_col='rating')
            print(f"Saving model to {load_path}...")
            model.save(load_path)
        
        print(f"Generating embeddings...")
        embeddings = model.get_embeddings()
        
        print(f"Generated {len(embeddings)} embeddings.")
        # Print shape of a sample embedding
        if embeddings:
            sample_key = list(embeddings.keys())[0]
            print(f"Shape of embedding for '{sample_key}': {embeddings[sample_key].shape}")

        # Evaluate
        print(f"Evaluating {name}...")
        metrics = calculate_metrics(model, train_df, test_df)
        print_metrics(metrics)
        
    except ImportError as e:
        print(f"Skipping {name} due to missing dependency: {e}")
    except Exception as e:
        print(f"An error occurred with {name}: {e}")
        import traceback
        traceback.print_exc()



--- Testing NCF ---
Found checkpoint at models\partial_ncf_model.pth. Attempting to load...
Model loaded successfully.
Generating embeddings...
Generated 98501 embeddings.
Shape of embedding for 'ml_user_1000': (64,)
Evaluating NCF...


Evaluating:   0%|          | 0/123229 [00:00<?, ?it/s]


Evaluation Results:
------------------------------
Precision@20: 0.0004
Precision@50: 0.0003
Precision@100: 0.0003
Precision@1000: 0.0001
HitRate@20: 0.0083
HitRate@50: 0.0175
HitRate@100: 0.0291
HitRate@1000: 0.1249
NDCG@20: 0.0030
NDCG@50: 0.0049
NDCG@100: 0.0067
NDCG@1000: 0.0180
MAP@20: 0.0016
MAP@50: 0.0019
MAP@100: 0.0021
MAP@1000: 0.0024
Diversity@20: 0.6694
Diversity@50: 0.6722
Diversity@100: 0.6656
Diversity@1000: 0.5884
Novelty@20: 13.4603
Novelty@50: 13.9575
Novelty@100: 14.3814
Novelty@1000: 15.8523
Serendipity@20: 1.0073
Serendipity@50: 1.0213
Serendipity@100: 1.0324
Serendipity@1000: 1.0646
AveragePopularity@20: 167.5091
AveragePopularity@50: 137.9193
AveragePopularity@100: 113.8691
AveragePopularity@1000: 47.8825
AUC (Global): 0.4458
AUC (Sampled): 0.4458
MeanRank: 7737.3482
MRR: 0.0025
------------------------------

--- Testing Cleora ---
Found checkpoint at models\partial_cleora_model.pth. Attempting to load...
Model loaded successfully.
Generating embeddings...
Gene

Evaluating:   0%|          | 0/123229 [00:00<?, ?it/s]


Evaluation Results:
------------------------------
Precision@20: 0.0000
Precision@50: 0.0000
Precision@100: 0.0001
Precision@1000: 0.0001
HitRate@20: 0.0000
HitRate@50: 0.0002
HitRate@100: 0.0076
HitRate@1000: 0.1249
NDCG@20: 0.0000
NDCG@50: 0.0000
NDCG@100: 0.0012
NDCG@1000: 0.0149
MAP@20: 0.0000
MAP@50: 0.0000
MAP@100: 0.0001
MAP@1000: 0.0004
Diversity@20: 0.9861
Diversity@50: 0.8833
Diversity@100: 0.5519
Diversity@1000: 0.0651
Novelty@20: 18.4220
Novelty@50: 18.1002
Novelty@100: 17.3655
Novelty@1000: 16.1001
Serendipity@20: 0.9483
Serendipity@50: 0.6594
Serendipity@100: 0.3308
Serendipity@1000: 0.0343
AveragePopularity@20: 1.1028
AveragePopularity@50: 1.5549
AveragePopularity@100: 29.7119
AveragePopularity@1000: 48.2355
AUC (Global): 0.4537
AUC (Sampled): 0.4539
MeanRank: 5842.1510
MRR: 0.0008
------------------------------
